Take-home-exercise-3

In this exercise, I build a model using F1 datasets and track each experiment run using MLflow. The goal is to predict the number of points a driver earns in a race.


In [0]:
base_path = "/Volumes/gr5069/raw/f1_data"

results = spark.read.csv(f"{base_path}/results.csv", header=True, inferSchema=True)
races = spark.read.csv(f"{base_path}/races.csv", header=True, inferSchema=True)
drivers = spark.read.csv(f"{base_path}/drivers.csv", header=True, inferSchema=True)
qualifying = spark.read.csv(f"{base_path}/qualifying.csv", header=True, inferSchema=True)
constructors = spark.read.csv(f"{base_path}/constructors.csv", header=True, inferSchema=True)
pit_stops = spark.read.csv(f"{base_path}/pit_stops.csv", header=True, inferSchema=True)

In [0]:
pit_counts = (
    pit_stops.groupBy("raceId", "driverId")
    .count()
    .withColumnRenamed("count", "pit_stop_count")
)

In [0]:
df = (
    results.join(
        races.select("raceId", "year", "round"),
        on="raceId",
        how="left"
    )
    .join(
        drivers.select("driverId", "dob"),
        on="driverId",
        how="left"
    )
    .join(
        qualifying.selectExpr("raceId", "driverId", "position as quali_position"),
        on=["raceId", "driverId"],
        how="left"
    )
    .join(
        pit_counts,
        on=["raceId", "driverId"],
        how="left"
    )
)

df = df.toPandas()

In [0]:
import pandas as pd
import numpy as np

df["dob"] = pd.to_datetime(df["dob"], errors="coerce")
df["age"] = df["year"] - df["dob"].dt.year

df["points"] = pd.to_numeric(df["points"], errors="coerce")
df["grid"] = pd.to_numeric(df["grid"], errors="coerce")
df["laps"] = pd.to_numeric(df["laps"], errors="coerce")
df["fastestLapSpeed"] = pd.to_numeric(df["fastestLapSpeed"], errors="coerce")
df["quali_position"] = pd.to_numeric(df["quali_position"], errors="coerce")
df["pit_stop_count"] = pd.to_numeric(df["pit_stop_count"], errors="coerce")

df["grid"] = df["grid"].replace(0, np.nan)
df["quali_position"] = df["quali_position"].fillna(df["grid"])
df["quali_position"] = df["quali_position"].fillna(df["quali_position"].median())
df["grid"] = df["grid"].fillna(df["grid"].median())
df["laps"] = df["laps"].fillna(df["laps"].median())
df["fastestLapSpeed"] = df["fastestLapSpeed"].fillna(df["fastestLapSpeed"].median())
df["age"] = df["age"].fillna(df["age"].median())
df["pit_stop_count"] = df["pit_stop_count"].fillna(0)

features = [
    "grid",
    "quali_position",
    "age",
    "laps",
    "fastestLapSpeed",
    "year",
    "round",
    "pit_stop_count"
]

model_df = df[features + ["points"]].dropna()

X = model_df[features]
y = model_df["points"]

Q1. Build a model with tunable hyperparameters

I built a Random Forest Regressor to predict the number of points a driver earns in a race. 
This is a regression task because the target variable (points) is numeric. Random Forest is suitable because it captures non-linear relationships and includes tunable hyperparameters such as:number of trees;maximum depth; minimum samples split; minimum samples leaf. I also included additional features such as pit stop count to improve model performance.

In [0]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

2. Create an experiment setup where - for each run - you log:

the hyperparameters used in the model
the model itself
every possible metric from the model you chose
at least two artifacts (plots, or csv files)

In [0]:
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import ParameterGrid

mlflow.set_experiment("/Users/hw3122@columbia.edu/take_home_exercise_3")

In [0]:
param_grid = {
    "n_estimators": [50, 100, 150, 200],
    "max_depth": [5, 10, 15],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

param_list = list(ParameterGrid(param_grid))[:10]

3.Track your MLFlow experiment and run at least 10 experiments with different parameters each



I run 10 experiments using different hyperparameter combinations to compare performance.

In [0]:
import os
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

best_rmse = float("inf")
best_params = None
best_run_id = None

artifact_dir = "/tmp/f1_mlflow_artifacts"
os.makedirs(artifact_dir, exist_ok=True)

for i, params in enumerate(param_list, start=1):
    with mlflow.start_run(run_name=f"rf_run_{i}") as run:

        # build and train model
        model = RandomForestRegressor(
            n_estimators=params["n_estimators"],
            max_depth=params["max_depth"],
            min_samples_split=params["min_samples_split"],
            min_samples_leaf=params["min_samples_leaf"],
            random_state=42
        )

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # metrics
        mse = mean_squared_error(y_test, y_pred)
        rmse = mse ** 0.5
        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        # log params and metrics
        mlflow.log_params(params)
        mlflow.log_metric("mse", mse)
        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("mae", mae)
        mlflow.log_metric("r2", r2)

        # log model
        mlflow.sklearn.log_model(model, "random_forest_model")

        # artifact 1: actual vs predicted scatter plot
        scatter_path = os.path.join(artifact_dir, f"actual_vs_predicted_run_{i}.png")
        plt.figure(figsize=(6, 6))
        plt.scatter(y_test, y_pred, alpha=0.5)
        plt.xlabel("Actual Points")
        plt.ylabel("Predicted Points")
        plt.title(f"Actual vs Predicted - Run {i}")
        plt.savefig(scatter_path, bbox_inches="tight")
        plt.close()
        mlflow.log_artifact(scatter_path)

        # artifact 2: feature importance plot
        importance_path = os.path.join(artifact_dir, f"feature_importance_run_{i}.png")
        importances = pd.Series(
            model.feature_importances_,
            index=X_train.columns
        ).sort_values(ascending=False)

        plt.figure(figsize=(8, 5))
        importances.plot(kind="bar")
        plt.title(f"Feature Importance - Run {i}")
        plt.ylabel("Importance")
        plt.savefig(importance_path, bbox_inches="tight")
        plt.close()
        mlflow.log_artifact(importance_path)

        # artifact 3: predictions csv
        pred_path = os.path.join(artifact_dir, f"predictions_run_{i}.csv")
        pred_df = pd.DataFrame({
            "actual_points": y_test.values,
            "predicted_points": y_pred
        })
        pred_df.to_csv(pred_path, index=False)
        mlflow.log_artifact(pred_path)

        # track best run
        if rmse < best_rmse:
            best_rmse = rmse
            best_params = params
            best_run_id = run.info.run_id

print("Best RMSE:", best_rmse)
print("Best Params:", best_params)
print("Best Run ID:", best_run_id)

In [0]:
print("Best RMSE:", best_rmse)
print("Best Params:", best_params)
print("Best Run ID:", best_run_id)

4.Select your best model run and explain why



I selected the best model based on the lowest RMSE. The best run achieved an RMSE of approximately 2.56, indicating that the model’s predictions are on average about 2.56 points away from the true values.
This run used the following hyperparameters:
n_estimators = 150；
max_depth = 5；
min_samples_split = 2；
min_samples_leaf = 1.

In addition to RMSE, I also examined MAE and R² to ensure consistent performance across different evaluation metrics. The selected model showed stable performance and no signs of overfitting.
MLflow made it easy to compare all runs by tracking parameters, metrics, models, and artifacts. Based on these results, I selected this run as the best model for predicting driver points.